# Resting-State Functional Connectivity & Connectome-Based Predictive Modeling

**Author:** Sanjana Singh  
**Topic:** fMRI connectivity analysis (HCP + AABC datasets)

Parcellation with Schaefer 400 / Glasser HCP-MMP1, Fisher-z connectivity matrices, lifespan ANOVA, and CPM prediction of Flanker executive function from resting-state edges (LOOCV, n=1,379).


## Setup & Imports

In [ ]:
import os, glob, re, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.lines import Line2D
import seaborn as sns
from scipy import stats
from scipy.stats import pearsonr, f_oneway
from statsmodels.stats.multitest import multipletests
import nibabel as nib

warnings.filterwarnings('ignore')
plt.rcParams.update({
    'figure.dpi': 150,
    'font.size': 10,
    'axes.titlesize': 12,
    'axes.labelsize': 10,
})

print("imports ok")


---
# Part I: Parcellation & Matrix Construction


## 1A. Load CIFTI Data & Schaefer Atlas

In [ ]:
# Load the preprocessed resting-state CIFTI file
cifti_path = 'rfMRI_REST1_LR_Atlas_hp2000_clean.dtseries.nii'   # adjust path as needed
cifti = nib.load(cifti_path)
data = cifti.get_fdata()          # shape: (1200, 91282)
print(f"CIFTI data shape: {data.shape}")
print(f"  Timepoints : {data.shape[0]}")
print(f"  Grayordinates: {data.shape[1]}")

TR = 0.72  # seconds
n_timepoints = data.shape[0]
print(f"  TR = {TR} s, total duration ~{n_timepoints * TR:.0f} s")


In [ ]:
# Load Schaefer 400-parcel atlas (17-network variant)
atlas_path = 'Schaefer2018_400Parcels_17Networks_order.dlabel.nii'  # adjust path
atlas_cifti = nib.load(atlas_path)
atlas_labels = atlas_cifti.get_fdata().squeeze().astype(int)  # (91282,)
print(f"Atlas labels shape: {atlas_labels.shape}")
print(f"Unique labels (incl. 0=medial wall): {len(np.unique(atlas_labels))}")

# Get label table for network assignment
label_table = atlas_cifti.header.get_axis(0)


## 1B. Parcellation: Extract ROI Time-Series

In [ ]:
# Map each parcel to a Yeo 7-network
# Schaefer 17-network parcel names contain the 7-network name, so map 17 -> 7 Yeo networks.

network_map_17to7 = {
    'Vis':   'Visual',
    'SomMot': 'SomatoMotor',
    'DorsAttn': 'DorsAttn',
    'SalVentAttn': 'SalVentAttn',
    'Limbic': 'Limbic',
    'Cont':  'Control',
    'Default': 'Default',
}

def get_yeo7_network(label_name):
    """Extract 7-network identity from a Schaefer 17-network parcel name."""
    for key, net in network_map_17to7.items():
        if key in label_name:
            return net
    return 'Unknown'

# Build parcel info dataframe
parcel_ids = sorted([l for l in np.unique(atlas_labels) if l != 0])
n_parcels = len(parcel_ids)
print(f"Number of parcels: {n_parcels}")

# Attempt to read label names from the CIFTI header
try:
    named_maps = list(atlas_cifti.header.get_index_map(0).named_maps)
    label_dict = list(named_maps[0].label_table.items())
    parcel_names = {idx: name for idx, (name, _) in enumerate(label_dict) if idx > 0}
except Exception:
    # fallback if the header parsing fails
    parcel_names = {i: f"Parcel_{i}" for i in parcel_ids}

parcel_info = []
for pid in parcel_ids:
    name = parcel_names.get(pid, f"Parcel_{pid}")
    net7 = get_yeo7_network(name)
    parcel_info.append({'parcel_id': pid, 'name': name, 'network': net7})

parcel_df = pd.DataFrame(parcel_info)
print("\nParcels per Yeo 7-network:")
print(parcel_df['network'].value_counts().sort_index())


In [ ]:
# Extract mean BOLD time-series per parcel
timeseries = np.zeros((n_timepoints, n_parcels))

for i, pid in enumerate(parcel_ids):
    mask = atlas_labels == pid
    timeseries[:, i] = data[:, mask].mean(axis=1)

print(f"ROI time-series matrix shape: {timeseries.shape}  (timepoints x parcels)")


## 1C. Time-Series Visualization

In [ ]:
# Z-score each ROI's time-series
ts_z = stats.zscore(timeseries, axis=0)

# Select 4 ROIs from each of 5 networks for plotting
plot_networks = ['Default', 'SalVentAttn', 'DorsAttn', 'Visual', 'SomatoMotor']
net_colors = {
    'Default':      '#E41A1C',
    'SalVentAttn':  '#984EA3',
    'DorsAttn':     '#4DAF4A',
    'Visual':       '#377EB8',
    'SomatoMotor':  '#FF7F00',
}

time_axis = np.arange(n_timepoints) * TR  # in seconds

fig, axes = plt.subplots(len(plot_networks), 1, figsize=(12, 10), sharex=True)
fig.suptitle('Resting-State BOLD Time-Series by Yeo Network\n(z-scored per ROI)',
             fontsize=14, fontweight='bold')

for ax, net in zip(axes, plot_networks):
    net_idx = parcel_df[parcel_df['network'] == net].index.tolist()
    chosen = net_idx[:4]  # first 4 parcels in that network
    for idx in chosen:
        ax.plot(time_axis, ts_z[:, idx], alpha=0.8, linewidth=0.6,
                color=net_colors[net],
                label=f"Parcel {parcel_ids[idx]}")
    ax.set_ylabel('BOLD (z)')
    ax.set_title(net, fontsize=11, fontweight='bold', color=net_colors[net])
    ax.legend(loc='upper right', fontsize=7, ncol=2)

axes[-1].set_xlabel('Time (seconds)')
plt.tight_layout()
plt.savefig('fig1_timeseries.png', dpi=200, bbox_inches='tight')
plt.show()
print("Figure 1 saved.")


## 1D. Connectivity Matrix Construction

In [ ]:
# Pearson correlation -> Fisher z
corr_matrix = np.corrcoef(timeseries.T)  # 400 x 400
np.fill_diagonal(corr_matrix, 0)

# Fisher r-to-z (clip to avoid +/-1)
corr_clipped = np.clip(corr_matrix, -0.9999, 0.9999)
z_matrix = np.arctanh(corr_clipped)
np.fill_diagonal(z_matrix, 0)

triu_idx = np.triu_indices(n_parcels, k=1)
z_upper = z_matrix[triu_idx]
print(f"Z-transformed matrix range: [{z_upper.min():.3f}, {z_upper.max():.3f}]")
print(f"Mean off-diagonal z: {z_upper.mean():.3f}")


## 1E. Network Structure Visualization

In [ ]:
# Reorder parcels by Yeo 7-network for block-diagonal view
network_order = ['Visual', 'SomatoMotor', 'DorsAttn', 'SalVentAttn',
                 'Limbic', 'Control', 'Default']

sorted_idx = []
boundaries = []
for net in network_order:
    net_parcels = parcel_df[parcel_df['network'] == net].index.tolist()
    boundaries.append(len(sorted_idx))
    sorted_idx.extend(net_parcels)
boundaries.append(len(sorted_idx))

z_ordered = z_matrix[np.ix_(sorted_idx, sorted_idx)]

# Plot
fig, ax = plt.subplots(figsize=(10, 9))
im = ax.imshow(z_ordered, cmap='RdBu_r', vmin=-0.6, vmax=0.6, aspect='equal')

# Network boundary lines and labels
for b in boundaries[1:-1]:
    ax.axhline(b - 0.5, color='k', linewidth=0.5, linestyle='--')
    ax.axvline(b - 0.5, color='k', linewidth=0.5, linestyle='--')

mid_points = [(boundaries[i] + boundaries[i+1]) / 2 for i in range(len(network_order))]
net_colors_list = ['#377EB8', '#FF7F00', '#4DAF4A', '#984EA3',
                   '#FFFF33', '#E41A1C', '#A65628']

ax.set_yticks(mid_points)
ax.set_yticklabels(network_order, fontsize=9)
ax.set_xticks(mid_points)
ax.set_xticklabels(network_order, fontsize=8, rotation=45, ha='right')
for tick, color in zip(ax.get_yticklabels(), net_colors_list):
    tick.set_color(color)
    tick.set_fontweight('bold')
for tick, color in zip(ax.get_xticklabels(), net_colors_list):
    tick.set_color(color)
    tick.set_fontweight('bold')

ax.set_ylabel('ROI (ordered by network)')
ax.set_xlabel('ROI (ordered by network)')
ax.set_title('Resting-State Functional Connectivity Matrix\n'
             'Schaefer 400-Parcel, Yeo 7-Network Order (Fisher z)',
             fontsize=13, fontweight='bold')

cbar = fig.colorbar(im, ax=ax, shrink=0.8)
cbar.set_label('Fisher z (r-to-z transformed)')

# Add network color patches in legend
legend_elements = [Line2D([0], [0], marker='s', color='w',
                          markerfacecolor=c, markersize=8, label=n)
                   for n, c in zip(network_order, net_colors_list)]
ax.legend(handles=legend_elements, title='Network', loc='lower right',
          fontsize=7, title_fontsize=8)

plt.tight_layout()
plt.savefig('fig2_connectivity_matrix.png', dpi=200, bbox_inches='tight')
plt.show()

# Within-network summary statistics
print("\nWithin-network mean z (SD):")
for i, net in enumerate(network_order):
    s, e = boundaries[i], boundaries[i+1]
    block = z_ordered[s:e, s:e]
    triu = block[np.triu_indices(e - s, k=1)]
    print(f"  {net:20s}: mean z = {triu.mean():.3f}, SD = {triu.std():.3f}")


---
# Part II: Lifespan Connectivity Analysis


## 2A. Data Loading and Organization

In [ ]:
# Load AABC Release 2 full-covariance connectivity CSVs
# Each CSV: one source ROI, covariance with all 379 target ROIs for all subjects
# The folder contains 379 CSV files.

data_dir = 'AABC_Release2_rfMRI_REST_FullCovarianceConnectivity'  # adjust path

csv_files = sorted(glob.glob(os.path.join(data_dir, '*.csv')))
print(f"Found {len(csv_files)} CSV files")

# Read the first file to get subject IDs and structure
df0 = pd.read_csv(csv_files[0])
print(f"Columns in first CSV: {df0.columns.tolist()[:6]} ...")
print(f"Shape: {df0.shape}")


In [ ]:
# Parse subject IDs and retain only V1 (first visit)
# Subject IDs may have suffixes like _V1, _V2, _V3

# subject ID column (usually the first one)
id_col = df0.columns[0]

def parse_subject(s):
    """Return (base_id, visit) from strings like 'HCA1234567_V1_MR'."""
    parts = str(s).split('_')
    base = parts[0]
    visit = 'V1'  # default
    for p in parts[1:]:
        if p.startswith('V') and p[1:].isdigit():
            visit = p
            break
    return base, visit

subjects_info = df0[id_col].apply(parse_subject)
df0['base_id'] = [s[0] for s in subjects_info]
df0['visit'] = [s[1] for s in subjects_info]

# Keep only V1
v1_mask = df0['visit'] == 'V1'
v1_subjects = df0.loc[v1_mask, id_col].tolist()
n_subjects = len(v1_subjects)
print(f"Subjects after V1 filtering: {n_subjects}")


In [ ]:
# Build the full subjects x 379 x 379 covariance array
n_rois = len(csv_files)
cov_array = np.zeros((n_subjects, n_rois, n_rois), dtype=np.float32)

for i, f in enumerate(csv_files):
    df_tmp = pd.read_csv(f)
    # Filter to V1 subjects (same order as v1_subjects)
    df_v1 = df_tmp[df_tmp[id_col].isin(v1_subjects)].copy()
    df_v1 = df_v1.set_index(id_col).loc[v1_subjects]  # ensure consistent order
    # The remaining columns are the 379 target ROIs
    roi_cols = [c for c in df_v1.columns if c not in ['base_id', 'visit']]
    cov_array[:, i, :len(roi_cols)] = df_v1[roi_cols].values.astype(np.float32)

    if (i + 1) % 50 == 0:
        print(f"  Loaded {i+1}/{n_rois} source ROIs ...")

print(f"\nFinal covariance array shape: {cov_array.shape}")


## 2B. Age Group Creation

In [ ]:
# Load demographics
demo_path = 'AABC_demographics.csv'   # adjust path
demo = pd.read_csv(demo_path)

# match ages to V1 subjects on base subject ID
demo['base_id'] = demo[demo.columns[0]].apply(lambda x: str(x).split('_')[0])

# base_id -> age
age_map = dict(zip(demo['base_id'], demo['age_open']))  # adjust column name

subject_bases = [parse_subject(s)[0] for s in v1_subjects]
ages = np.array([age_map.get(b, np.nan) for b in subject_bases])
print(f"Age range: {np.nanmin(ages):.0f} - {np.nanmax(ages):.0f}")
print(f"Mean age: {np.nanmean(ages):.1f} (SD = {np.nanstd(ages):.1f})")

# Define age groups
age_bins = [36, 50, 65, 80, 120]   # 36-50, 50-65, 65-80, 80+
age_labels = ['36-50', '50-65', '65-80', '80+']
age_groups = pd.cut(ages, bins=age_bins, labels=age_labels, right=False)

print("\nSample sizes per group:")
for lab in age_labels:
    print(f"  {lab}: N = {np.sum(age_groups == lab)}")


## 2C. Grand-Average Connectivity Matrices

In [ ]:
# Compute mean connectivity matrix per age group
group_means = {}
for lab in age_labels:
    idx = np.where(age_groups == lab)[0]
    group_means[lab] = cov_array[idx].mean(axis=0)
    print(f"  {lab}: mean covariance = {group_means[lab].mean():.2f}")

# Plot
fig, axes = plt.subplots(2, 2, figsize=(14, 12))
fig.suptitle('Grand-Average Functional Connectivity by Age Group\n'
             '(Glasser HCP-MMP1 379 ROIs; yellow dashed = cortical/subcortical boundary)',
             fontsize=13, fontweight='bold')

# Shared colorbar limits
vmax = max(np.abs(group_means[lab]).max() for lab in age_labels)
vmin = -vmax

sample_sizes = {lab: int(np.sum(age_groups == lab)) for lab in age_labels}

for ax, lab in zip(axes.flat, age_labels):
    im = ax.imshow(group_means[lab], cmap='RdBu_r', vmin=-vmax*0.8, vmax=vmax*0.8,
                   aspect='equal')
    ax.set_title(f"Age {lab}  (N = {sample_sizes[lab]})", fontsize=11, fontweight='bold')
    ax.set_xlabel('Target ROI index')
    ax.set_ylabel('Source ROI index')
    # Mark cortical/subcortical boundary (ROI 360)
    ax.axhline(359.5, color='yellow', linewidth=1, linestyle='--')
    ax.axvline(359.5, color='yellow', linewidth=1, linestyle='--')

fig.subplots_adjust(right=0.88)
cbar_ax = fig.add_axes([0.90, 0.15, 0.02, 0.7])
cbar = fig.colorbar(im, cax=cbar_ax)
cbar.set_label('Mean covariance (a.u.)')

plt.savefig('fig3_grand_average_matrices.png', dpi=200, bbox_inches='tight')
plt.show()


## 2D. Contrast Analysis: One-Way ANOVA

In [ ]:
# One-way ANOVA at each edge across the 4 age groups
n_rois_aabc = cov_array.shape[1]  # 379
F_matrix = np.zeros((n_rois_aabc, n_rois_aabc))
p_matrix = np.ones((n_rois_aabc, n_rois_aabc))

# Group indices
group_idx = {lab: np.where(age_groups == lab)[0] for lab in age_labels}

print("Running ANOVA across all edges ... (this may take a minute)")
for i in range(n_rois_aabc):
    for j in range(i + 1, n_rois_aabc):
        groups_data = [cov_array[group_idx[lab], i, j] for lab in age_labels]
        F_val, p_val = f_oneway(*groups_data)
        F_matrix[i, j] = F_val
        F_matrix[j, i] = F_val
        p_matrix[i, j] = p_val
        p_matrix[j, i] = p_val
    if (i + 1) % 50 == 0:
        print(f"  Processed row {i+1}/{n_rois_aabc}")

# Bonferroni correction
n_edges = n_rois_aabc * (n_rois_aabc - 1) // 2
bonf_thresh = 0.05 / n_edges
n_sig = np.sum(p_matrix[np.triu_indices(n_rois_aabc, k=1)] < bonf_thresh)
pct_sig = 100 * n_sig / n_edges

print(f"\nTotal unique edges: {n_edges}")
print(f"Bonferroni threshold: p < {bonf_thresh:.2e}")
print(f"Significant edges: {n_sig} ({pct_sig:.2f}%)")


In [ ]:
# Plot F-statistic matrix
fig, ax = plt.subplots(figsize=(10, 9))
im = ax.imshow(F_matrix, cmap='hot', aspect='equal')
ax.set_title('One-Way ANOVA: F-Statistic Matrix\n'
             '(Age groups: 36-50 | 50-65 | 65-80 | 80+)',
             fontsize=12, fontweight='bold')
ax.set_xlabel('Target ROI index')
ax.set_ylabel('Source ROI index')
# Cortical/subcortical boundary
ax.axhline(359.5, color='teal', linewidth=1, linestyle='--')
ax.axvline(359.5, color='teal', linewidth=1, linestyle='--')
cbar = fig.colorbar(im, ax=ax, shrink=0.8)
cbar.set_label('F-statistic')

plt.tight_layout()
plt.savefig('fig4_anova_fstat.png', dpi=200, bbox_inches='tight')
plt.show()


In [ ]:
# Plot -log10(p) matrix
neglog_p = -np.log10(np.clip(p_matrix, 1e-300, 1))
np.fill_diagonal(neglog_p, 0)

bonf_neglog = -np.log10(bonf_thresh)

fig, ax = plt.subplots(figsize=(10, 9))
im = ax.imshow(neglog_p, cmap='magma', aspect='equal')
ax.set_title('One-Way ANOVA: -log10(p) Matrix\n'
             '(brighter = stronger age difference)',
             fontsize=12, fontweight='bold')
ax.set_xlabel('Target ROI index')
ax.set_ylabel('Source ROI index')
ax.axhline(359.5, color='teal', linewidth=1, linestyle='--')
ax.axvline(359.5, color='teal', linewidth=1, linestyle='--')

cbar = fig.colorbar(im, ax=ax, shrink=0.8)
cbar.set_label('-log10(p)')
# Mark Bonferroni threshold on colorbar
cbar.ax.axhline(bonf_neglog, color='#00FF00', linewidth=2, linestyle='--')
cbar.ax.text(1.5, bonf_neglog, f'Bonferroni\np<{bonf_thresh:.0e}',
             fontsize=7, color='#00FF00', va='center')

plt.tight_layout()
plt.savefig('fig5_anova_pval.png', dpi=200, bbox_inches='tight')
plt.show()


---
# Part III (Bonus): Connectome-Based Predictive Modeling (CPM)


## 3A. Load Behavioral Variable

In [ ]:
# Load Flanker executive function scores
# Using the same demographics / behavioral file
# Target variable: tlbx_fica_rawscore (Flanker Inhibitory Control & Attention)

behav_col = 'tlbx_fica_rawscore'   # adjust column name if needed

flanker_scores = np.array([
    demo.loc[demo['base_id'] == b, behav_col].values[0]
    if b in demo['base_id'].values and not pd.isna(
        demo.loc[demo['base_id'] == b, behav_col].values[0])
    else np.nan
    for b in subject_bases
])

valid_mask = ~np.isnan(flanker_scores)
n_valid = valid_mask.sum()
print(f"Subjects with valid Flanker scores: {n_valid}")
print(f"Flanker range: {np.nanmin(flanker_scores):.1f} - {np.nanmax(flanker_scores):.1f}")
print(f"Mean = {np.nanmean(flanker_scores):.2f}, SD = {np.nanstd(flanker_scores):.2f}")

# Filter arrays
cov_valid = cov_array[valid_mask]
flanker_valid = flanker_scores[valid_mask]


## 3B. CPM: Leave-One-Out Cross-Validation

In [ ]:
# Extract upper triangle edges
triu_i, triu_j = np.triu_indices(n_rois_aabc, k=1)
n_edges_total = len(triu_i)
print(f"Edges per subject: {n_edges_total}")

# flatten to subjects x edges
edges_all = np.zeros((n_valid, n_edges_total), dtype=np.float32)
for s in range(n_valid):
    edges_all[s, :] = cov_valid[s][triu_i, triu_j]

print(f"Edge matrix shape: {edges_all.shape}")


In [ ]:
# CPM LOOCV
p_thresh = 0.01
n_subj = n_valid

pred_pos = np.zeros(n_subj)
pred_neg = np.zeros(n_subj)
pred_comb = np.zeros(n_subj)

pos_edge_counts = []
neg_edge_counts = []

print(f"Running {n_subj}-fold LOOCV ...")

for i in range(n_subj):
    # Train/test split
    train_mask = np.ones(n_subj, dtype=bool)
    train_mask[i] = False

    train_edges = edges_all[train_mask]
    train_behav = flanker_valid[train_mask]
    test_edges = edges_all[i]

    # Edge selection: correlate each edge with behavior in training set
    r_vals = np.zeros(n_edges_total)
    p_vals = np.ones(n_edges_total)
    for e in range(n_edges_total):
        r_vals[e], p_vals[e] = pearsonr(train_edges[:, e], train_behav)

    # Select significant edges
    pos_mask = (p_vals < p_thresh) & (r_vals > 0)
    neg_mask = (p_vals < p_thresh) & (r_vals < 0)
    pos_edge_counts.append(pos_mask.sum())
    neg_edge_counts.append(neg_mask.sum())

    # Network strengths
    train_pos_str = train_edges[:, pos_mask].sum(axis=1) if pos_mask.any() else np.zeros(n_subj - 1)
    train_neg_str = train_edges[:, neg_mask].sum(axis=1) if neg_mask.any() else np.zeros(n_subj - 1)
    test_pos_str = test_edges[pos_mask].sum() if pos_mask.any() else 0
    test_neg_str = test_edges[neg_mask].sum() if neg_mask.any() else 0

    # Positive model
    if pos_mask.any():
        b_pos = np.polyfit(train_pos_str, train_behav, 1)
        pred_pos[i] = np.polyval(b_pos, test_pos_str)

    # Negative model
    if neg_mask.any():
        b_neg = np.polyfit(train_neg_str, train_behav, 1)
        pred_neg[i] = np.polyval(b_neg, test_neg_str)

    # Combined model (pos - neg)
    train_comb_str = train_pos_str - train_neg_str
    test_comb_str = test_pos_str - test_neg_str
    b_comb = np.polyfit(train_comb_str, train_behav, 1)
    pred_comb[i] = np.polyval(b_comb, test_comb_str)

    if (i + 1) % 200 == 0:
        print(f"  Fold {i+1}/{n_subj}")

print("\nDone.")
print(f"Mean positive edges selected: {np.mean(pos_edge_counts):.0f}")
print(f"Mean negative edges selected: {np.mean(neg_edge_counts):.0f}")


## 3C. CPM Results

In [ ]:
# Evaluate predictions
r_pos, p_pos = pearsonr(flanker_valid, pred_pos)
r_neg, p_neg = pearsonr(flanker_valid, pred_neg)
r_comb, p_comb = pearsonr(flanker_valid, pred_comb)

print("CPM Results:")
print(f"  Positive model:  r = {r_pos:+.3f}, p = {p_pos:.3e}")
print(f"  Negative model:  r = {r_neg:+.3f}, p = {p_neg:.3e}")
print(f"  Combined model:  r = {r_comb:+.3f}, p = {p_comb:.3e}")


In [ ]:
# Scatter plots
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('CPM LOOCV: Predicted vs Observed Flanker Executive Function\n'
             '(AABC Release 2 | Glasser 379-ROI covariance FC | p<0.01 edge threshold)',
             fontsize=12, fontweight='bold')

models = [
    ('Positive Network Model', pred_pos, r_pos, p_pos),
    ('Negative Network Model', pred_neg, r_neg, p_neg),
    ('Combined Network Model\n(Pos - Neg strength)', pred_comb, r_comb, p_comb),
]

for ax, (title, preds, r_val, p_val) in zip(axes, models):
    ax.scatter(flanker_valid, preds, alpha=0.3, s=15, color='#FF7F0E', edgecolors='none')

    # Fit line
    m, b = np.polyfit(flanker_valid, preds, 1)
    x_line = np.linspace(flanker_valid.min(), flanker_valid.max(), 100)
    ax.plot(x_line, m * x_line + b, 'r--', linewidth=1.5, label='fit')

    ax.set_title(title, fontsize=10, fontweight='bold')
    ax.set_xlabel('Observed Flanker Score')
    ax.set_ylabel('Predicted Flanker Score')
    ax.legend(fontsize=8, loc='lower right')

    # Annotate r, p
    sig_str = '***' if p_val < 0.001 else ('**' if p_val < 0.01 else ('*' if p_val < 0.05 else 'n.s.'))
    ax.text(0.05, 0.95, f'r = {r_val:+.3f}\np = {p_val:.2e}',
            transform=ax.transAxes, fontsize=9, verticalalignment='top',
            bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

plt.tight_layout()
plt.savefig('fig6_cpm_results.png', dpi=200, bbox_inches='tight')
plt.show()


## References

- Glasser, M. F., et al. (2016). A multi-modal parcellation of human cerebral cortex. *Nature*, 536(7615), 171-178.
- Schaefer, A., et al. (2018). Local-global parcellation of the human cerebral cortex from intrinsic functional connectivity MRI. *Cerebral Cortex*, 28(9), 3095-3114.
- Shen, X., et al. (2017). Using connectome-based predictive modeling to predict individual behavior from brain connectivity. *Nature Protocols*, 12(3), 506-518.
- Van Essen, D. C., et al. (2013). The WU-Minn Human Connectome Project: An overview. *NeuroImage*, 80, 62-79.
- Yeo, B. T., et al. (2011). The organization of the human cerebral cortex estimated by intrinsic functional connectivity. *Journal of Neurophysiology*, 106(3), 1125-1165.
